In [1]:
from pathlib import Path
import random
import shutil

# If notebook is in /notebooks
PROJECT_ROOT = Path.cwd().parent

CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

TRAIN_DIR = SPLITS_DIR / "train"
VAL_DIR   = SPLITS_DIR / "val"
TEST_DIR  = SPLITS_DIR / "test"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Split ratios
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9

SEED = 42
random.seed(SEED)

def list_images(folder: Path):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])

def safe_copy(src: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_path = dst_dir / src.name

    # avoid overwrite
    if dst_path.exists():
        stem, ext = src.stem, src.suffix
        i = 1
        while True:
            candidate = dst_dir / f"{stem}_{i}{ext}"
            if not candidate.exists():
                dst_path = candidate
                break
            i += 1

    shutil.copy2(src, dst_path)
    return dst_path

# (Optional) clear old splits
if SPLITS_DIR.exists():
    print("Removing old splits directory:", SPLITS_DIR)
    shutil.rmtree(SPLITS_DIR)

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

summary = []

class_folders = sorted([d for d in CLEAN_DIR.iterdir() if d.is_dir()])

for class_dir in class_folders:
    imgs = list_images(class_dir)
    n = len(imgs)

    if n < 5:
        print(f"WARNING: very small class '{class_dir.name}' with n={n}")

    random.shuffle(imgs)

    n_train = int(n * TRAIN_FRAC)
    n_val   = int(n * VAL_FRAC)
    # remainder goes to test (ensures all images used)
    n_test  = n - n_train - n_val

    train_imgs = imgs[:n_train]
    val_imgs   = imgs[n_train:n_train + n_val]
    test_imgs  = imgs[n_train + n_val:]

    # Ensure folders exist
    (TRAIN_DIR / class_dir.name).mkdir(parents=True, exist_ok=True)
    (VAL_DIR / class_dir.name).mkdir(parents=True, exist_ok=True)
    (TEST_DIR / class_dir.name).mkdir(parents=True, exist_ok=True)

    for p in train_imgs:
        safe_copy(p, TRAIN_DIR / class_dir.name)
    for p in val_imgs:
        safe_copy(p, VAL_DIR / class_dir.name)
    for p in test_imgs:
        safe_copy(p, TEST_DIR / class_dir.name)

    summary.append((class_dir.name, n_train, n_val, n_test, n))

print("\nSplit complete.")
print("Saved to:", SPLITS_DIR)



Split complete.
Saved to: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification\data\splits


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder: Path) -> int:
    return sum(1 for p in folder.glob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

for split in ["train", "val", "test"]:
    split_dir = SPLITS_DIR / split
    total = 0
    print(f"\nCounts in {split}:")
    for class_dir in sorted([d for d in split_dir.iterdir() if d.is_dir()]):
        c = count_images(class_dir)
        total += c
        print(f"  {class_dir.name}: {c}")
    print(f"TOTAL {split}: {total}")



Counts in train:
  class_01_motorcycle: 203
  class_02_passenger_car: 690
  class_03_four_tire_single_unit: 49
  class_04_bus: 182
  class_05_two_axle_six_tire_single_unit: 60
  class_06_three_axle_single_unit: 88
  class_07_four_or_more_axle_single_unit: 25
  class_08_four_or_less_axle_single_trailer: 11
  class_09_five_axle_tractor_semitrailer: 28
  class_10_six_or_more_axle_single_trailer: 26
  class_11_five_or_less_axle_multi_trailer: 27
  class_13_seven_or_more_axle_multi_trailer: 12
TOTAL train: 1401

Counts in val:
  class_01_motorcycle: 43
  class_02_passenger_car: 148
  class_03_four_tire_single_unit: 10
  class_04_bus: 39
  class_05_two_axle_six_tire_single_unit: 12
  class_06_three_axle_single_unit: 18
  class_07_four_or_more_axle_single_unit: 5
  class_08_four_or_less_axle_single_trailer: 2
  class_09_five_axle_tractor_semitrailer: 6
  class_10_six_or_more_axle_single_trailer: 5
  class_11_five_or_less_axle_multi_trailer: 5
  class_13_seven_or_more_axle_multi_trailer: 2
TO